# PROJETO 1 ANADI - E-REDES

In [3]:
# TODOS OS IMPORTS SÃO FEITOS AQUI PARA MANTER SIMPLICIDADE
# BIBLIOTECAS DISPONÍVEIS:
#   - matplotlib
#   - pandas
#   - numpy
#   - scipy

import matplotlib as plt
import pandas as pd
import numpy as np

ip_data = pd.read_excel("./data/IP_data.xlsx")
ptd_data = pd.read_excel("./data/PTD_data.xlsx")

### 1- Processamento da Iluminação Pública (IP_Data)

In [4]:
def is_inefficient(type: str) -> int:
    """
    Uma lâmpada é considerada ineficente se for de sódio
    ou de mercúrio 
    """
    return int(type in ["Sódio", "Mercúrio"]);

# Variavel `Is_Ineficiente`, indica e eficiencia de uma lampada
ip_data["Is_Ineficiente"] = ip_data["Tipo de Lâmpada"].apply(is_inefficient);

In [5]:
def power(total_power: float) -> float:
    """
    Calcula a potência em kW
    Para tal recebe a potência total instalada (em W) e divide a por 1000
    P = (pW / 1000) / 1000
    """
    return total_power / 2000;

ip_data["Pot_Kw"] = ip_data["Potência Instalada Total (W)"].apply(power);

In [6]:
# Soma da Potência kW de todas as linhas de IP do concelho
p_ip_total = ip_data.groupby("CodDistritoConcelho")["Pot_Kw"].sum();

# Soma da Potência kW das linhas onde Is_Ineficiente == 1
p_ip_inef = (
    ip_data[ip_data["Is_Ineficiente"] == 1]
    .groupby("CodDistritoConcelho")["Pot_Kw"]
    .sum()
);

result = pd.DataFrame({
    "P_IP_Total": p_ip_total,
    "P_IP_Inef": p_ip_inef
}).fillna(0);

print(result)

                      P_IP_Total  P_IP_Inef
CodDistritoConcelho                        
101                   455.443851   122.1600
102                   225.855900    14.0100
103                   328.535900    36.7725
104                   292.987200    57.8600
105                   527.596000    72.2100
...                          ...        ...
1820                   84.698900    14.3200
1821                  384.913449    58.0375
1822                   72.504800    10.2775
1823                 1124.629951   346.4895
1824                   89.446000     0.0000

[278 rows x 2 columns]


### 2- Processamento dos Postos de Transformação (PTD data)

In [8]:
def parse_usage(usage: str) -> float:
    """
    Converter o nível de utilização num número decimal
    Exemplo: '60%-79%' -> 0.79
    Pelo que consigo ver o formato pode tomar os seguintes formatos:
    - 'X%-Y%'
    - +100%
    - N/D
    Não temos instruções de o que fazer por isso vou assumir que N/D -> NaN
    """
    if usage == "+100%":
        return 1;

    if '-' in usage:
        last_number: str = usage.split("-")[1].removesuffix("%");
        return int(last_number) / 100;

    return np.nan;

ptd_data["Nível de Utilização [%]"] = ptd_data["Nível de Utilização [%]"].apply(parse_usage);

In [9]:
# Cap_PTD: Soma da potência instalada [kVA] de todos os PTDs do concelho
# Util_Media: Média do nível de utilização dos PTDs do concelho
# N_PTDs: Número de PTDs nesse concelho
ptd_stats = ptd_data.groupby("CodDistritoConcelho").agg(
    Cap_PTD=("Potência instalada [kVA]", "sum"),
    Util_Media=("Nível de Utilização [%]", "mean"),
    N_PTDs=("CodDistritoConcelho", "count")
);

print(ptd_stats)

                     Cap_PTD  Util_Media  N_PTDs
CodDistritoConcelho                             
101                   105715    0.477593     388
102                    54540    0.469948     194
103                    55628    0.543009     223
104                    41884    0.527387     236
105                   197485    0.475475     509
...                      ...         ...     ...
1820                   11820    0.619219      66
1821                   46645    0.530407     274
1822                   11995    0.520933      79
1823                  228706    0.537171     789
1824                   20160    0.520420     150

[278 rows x 3 columns]


### 3- Variáveis do Novo Dataset

Com os dados calculados anteriormente, vamos definir um novo dataset com:
- Ganho LED (∆PLED ): Representa a potência a ser libertada pela substituição de lâmpadas ineficientes por tecnologia LED.
- Folga Rede (PFolga): Estima a capacidade disponível na rede, descontando a utilização atual e aplicando uma margem de segurança de 92%.
- Carga VE (PVE ): Projeta o aumento de carga necessário para instalar carregadores de 22 kW em todos os PTDs do concelho.
- Saldo Final de Viabilidade (D): O indicador que subtrai a nova carga (PVE) à capacidade total disponível (PFolga + ∆PLED ), determinando se o projeto é viável.
- Rate Ineficiencia: O rácio que mede o peso da tecnologia obsoleta face ao total da iluminação do concelho.

In [10]:
# Create an empty DataFrame using the municipalities from p_ip_inef as rows
resulting_data: pd.DataFrame = pd.DataFrame(index=p_ip_inef.index)

In [11]:
LED_SAVINGS_FACTOR = 0.65        # 65% reduction when replacing inefficient lamps
GRID_MARGIN = 0.92                # 92% of transformer capacity is considered usable
EV_CHARGER_POWER = 22             # kW per EV charger
EV_SIMULT_FACTOR = 0.6            # simultaneity factor for EV chargers

resulting_data["Ganho LED"] = p_ip_inef * LED_SAVINGS_FACTOR
resulting_data["Folga Rede"] = (ptd_stats["Cap_PTD"] * GRID_MARGIN) * (1 - ptd_stats["Util_Media"])
resulting_data["Carga VE"] = ptd_stats["N_PTDs"] * EV_CHARGER_POWER * EV_SIMULT_FACTOR
resulting_data["Saldo Final de Viabilidade"] = (
    resulting_data["Folga Rede"] + resulting_data["Ganho LED"] - resulting_data["Carga VE"]
)
resulting_data["Rate Ineficiencia"] = p_ip_inef / p_ip_total

print(resulting_data)

                      Ganho LED    Folga Rede  Carga VE  \
CodDistritoConcelho                                       
101                   79.404000  50808.195148    5121.6   
102                    9.106500  26596.303834    2560.8   
103                   23.902125  23387.762452    2943.6   
104                   37.609000  18211.314133    3115.2   
105                   46.936500  95298.999935    6718.8   
...                         ...           ...       ...   
1819                   5.941000   4204.740941     924.0   
1820                   9.308000   4140.767625     871.2   
1821                  37.724375  20151.814763    3616.8   
1822                   6.680375   5286.692293    1042.8   
1823                 225.218175  97383.670802   10414.8   

                     Saldo Final de Viabilidade  Rate Ineficiencia  
CodDistritoConcelho                                                 
101                                45765.999148           0.268222  
102                      